# Baseline models on the Crunchbase startup outcome dataset

Loads the models trained by `python -m src.models.train` and reports their saved CV and held-out test metrics side by side. Run `python -m src.models.train` first if `models/` is empty.

In [1]:
import json
from pathlib import Path

import pandas as pd

MODEL_NAMES = [
    'dummy_most_frequent',
    'logistic_regression_full',
    'logistic_regression_clean',
    'hist_gradient_boosting_full',
    'hist_gradient_boosting_clean',
]

metadata = {name: json.loads(Path(f'../models/{name}_metadata.json').read_text()) for name in MODEL_NAMES}
len(metadata)

5

## Metrics table (test = time-separated holdout, CV = repeated stratified CV on train)

In [2]:
rows = []
for name, m in metadata.items():
    rows.append({
        'model': name,
        'feature_set': m['feature_set'],
        'cv_roc_auc_mean': m['cv']['roc_auc_mean'],
        'cv_roc_auc_std': m['cv']['roc_auc_std'],
        'test_roc_auc': m['test']['roc_auc'],
        'test_average_precision': m['test']['average_precision'],
        'test_brier': m['test']['brier_score'],
        'threshold': m['threshold'],
        'test_f0.5': m['test']['f0.5'],
    })
table = pd.DataFrame(rows).set_index('model')
table

,feature_set,cv_roc_auc_mean,cv_roc_auc_std,test_roc_auc,test_average_precision,test_brier,threshold,test_f0.5
model,,,,,,,,
dummy_most_frequent,full,0.500000,0.000000,0.500000,0.267092,0.732908,0.01,0.312968
logistic_regression_full,full,0.777745,0.009397,0.849554,0.652174,0.135580,0.63,0.570488
logistic_regression_clean,clean,0.727176,0.009331,0.802165,0.546296,0.154181,0.60,0.467236
hist_gradient_boosting_full,full,0.787950,0.009385,0.833938,0.656424,0.144387,0.65,0.604503
hist_gradient_boosting_clean,clean,0.736337,0.009551,0.798336,0.547562,0.163491,0.63,0.451486


## CV vs. test: why they differ

CV metrics are computed on repeated stratified folds *within* the train cohort (same era, positive rate 59.83%). The test metric is a single evaluation on the time-separated holdout (a later era, positive rate 26.71%). The gap between the two numbers reflects genuine distribution shift across time, not just sampling noise - this is the realistic deployment scenario (scoring newer companies), so the test number is the one that matters for expectations, not the CV number.

In [3]:
table[['cv_roc_auc_mean', 'test_roc_auc']].assign(
    gap=lambda d: d['test_roc_auc'] - d['cv_roc_auc_mean']
)

,cv_roc_auc_mean,test_roc_auc,gap
model,,,
dummy_most_frequent,0.500000,0.500000,0.000000
logistic_regression_full,0.777745,0.849554,0.071810
logistic_regression_clean,0.727176,0.802165,0.074989
hist_gradient_boosting_full,0.787950,0.833938,0.045988
hist_gradient_boosting_clean,0.736337,0.798336,0.061999


## Clean vs. full: the leakage finding

In [4]:
for family in ['logistic_regression', 'hist_gradient_boosting']:
    full_auc = metadata[f'{family}_full']['test']['roc_auc']
    clean_auc = metadata[f'{family}_clean']['test']['roc_auc']
    print(f'{family}: full={full_auc:.4f} clean={clean_auc:.4f} gap={full_auc - clean_auc:.4f}')

logistic_regression: full=0.8496 clean=0.8022 gap=0.0474
hist_gradient_boosting: full=0.8339 clean=0.7983 gap=0.0356


## Figures

See `../reports/figures/roc_comparison.png`, `pr_comparison.png`, `calibration_comparison.png` for the full ROC/PR/calibration curves across all 5 models.